<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/torch/easy/quantize-lm/quantize-language-model.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("quantize-lm", ...)`


## Problem: Quantize Your Language Model

### Problem Statement
Implement a small LSTM **language model** and apply **post-training dynamic quantization** to make it smaller and cheaper to run on CPU.

### Background: what dynamic quantization does
Quantization stores a float tensor `x` as 8-bit integers `q` using an affine map with a **scale** `s` and a **zero point** `z`:

$$q = \mathrm{clamp}\big(\mathrm{round}(x / s) + z,\ q_{\min},\ q_{\max}\big), \qquad \hat{x} = s\,(q - z)$$

Within the clamp range, the rounding error per element is at most `s / 2`.

- **Weights** of the module types you list (here `nn.Linear` and `nn.LSTM`) are converted to `qint8` **once, ahead of time**. The default config is per-tensor and symmetric: `z = 0` and `s = max|w| / 127.5`. Weight storage shrinks by 4×.
- **Activations** stay in float between layers. Inside each quantized layer, the input is quantized **on the fly**, with `s` and `z` computed from that batch's min and max. This is what *dynamic* refers to. The matmul runs in int8 with int32 accumulation, and the result is dequantized back to float. No calibration data is needed.
- **Not quantized**: any module type you did not list. Here that is `nn.Embedding`, which would need a weight-only config, and the softmax.
- **Static quantization** also fixes the activations' `s` and `z` ahead of time, by running observers over representative calibration data, and keeps activations in int8 between layers. It is more work and usually faster, and it is the standard choice for CNNs. Dynamic quantization suits models whose cost is dominated by loading weights, such as LSTMs and Transformer `Linear` layers at small batch sizes on CPU.
- **Speed is not guaranteed.** Int8 kernels and smaller weights help on CPUs with fast int8 support. For a model this small, quantizing activations on every call can cancel out the gain, so measure on your own hardware.

> `torch.ao.quantization` is deprecated in recent PyTorch releases and prints a `DeprecationWarning`. Its replacement is the [torchao](https://github.com/pytorch/ao) library (`torchao.quantization.quantize_`). The concepts carry over unchanged. This notebook uses the eager API, which still ships with PyTorch.

### Requirements
1. **Define the Language Model**: `LanguageModel(vocab_size, embed_size, hidden_size, num_layers)`.
   - **Embedding layer**: converts input tokens into dense vectors.
   - **LSTM layer** (`batch_first=True`): processes the embedded sequence.
   - **Fully connected layer**: maps the LSTM output at the **last time step** to one score per vocabulary token.
   - **Softmax**: turns the scores into a probability distribution over the vocabulary. The output shape is `(batch, vocab_size)`. In practice you would return logits and train with `nn.CrossEntropyLoss`. Because this model returns probabilities, training below uses `NLLLoss` on their log.
2. **Apply dynamic quantization** to the `nn.Linear` and `nn.LSTM` modules with `quantize_dynamic`, using `dtype=torch.qint8`.
3. **Compare** the quantized model with the float model: the size of the saved weights, how often they agree on the next-token prediction, and the largest difference between their probabilities.
4. **Look inside**: recompute the scale and the int8 values of the quantized `fc` weight from the float weight, and check that they match what PyTorch stored.

In [ ]:
import io

import torch
import torch.nn as nn
import torch.optim as optim
from torch.ao.quantization import quantize_dynamic

# Dynamic quantization needs a quantized backend. On Apple Silicon (and some other
# builds) the default engine is 'none', so pick one this torch build supports.
if torch.backends.quantized.engine == "none":
    engines = [e for e in torch.backends.quantized.supported_engines if e != "none"]
    if not engines:
        raise RuntimeError("this torch build has no quantized backend (fbgemm/qnnpack)")
    torch.backends.quantized.engine = engines[0]
print(f"Quantized engine: {torch.backends.quantized.engine}")

In [ ]:
# TODO: Define a simple Language Model (an LSTM-based model)
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super().__init__()
        ...

    def forward(self, x):
        ...

In [ ]:
# Create synthetic training data
torch.manual_seed(42)
vocab_size = 50
seq_length = 10
batch_size = 32
X_train = torch.randint(0, vocab_size, (batch_size, seq_length))  # Random integer input
y_train = torch.randint(0, vocab_size, (batch_size,))  # Random target words

# Initialize the model, loss function, and optimizer
embed_size = 64
hidden_size = 128
num_layers = 2
model = LanguageModel(vocab_size, embed_size, hidden_size, num_layers)

# The model ends in a softmax, so it returns probabilities, not logits.
# nn.CrossEntropyLoss would apply a second softmax on top and the loss would barely
# move, so train with NLLLoss on the log-probabilities instead.
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Training loop
epochs = 5
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(torch.log(output.clamp_min(1e-9)), y_train)
    loss.backward()
    optimizer.step()

    # Log progress every epoch
    print(f"Epoch [{epoch + 1}/{epochs}] - Loss: {loss.item():.4f}")

In [ ]:
# Quantize the trained model: int8 weights for every nn.Linear and nn.LSTM
model.eval()
# TODO: apply dynamic quantization (returns a quantized copy; `model` stays float)
quantized_model = ...

# Save the float weights. A dynamically quantized state_dict stores the LSTM's packed
# weights as torch.ScriptObject, which torch.load(weights_only=True) (the default
# since PyTorch 2.6) refuses to unpickle. Dynamic quantization is deterministic, so
# quantizing the float weights again after loading rebuilds the same model.
torch.save(model.state_dict(), "language_model_fp32.pth")

In [ ]:
# Load the float weights, then apply the same dynamic quantization again
quantized_model = LanguageModel(vocab_size, embed_size, hidden_size, num_layers)
quantized_model.load_state_dict(torch.load("language_model_fp32.pth", weights_only=True))
quantized_model.eval()
# TODO: quantize it exactly as above
quantized_model = ...

In [ ]:
# Testing the quantized model on a sample input
test_input = torch.randint(0, vocab_size, (1, seq_length))
with torch.no_grad():
    prediction = quantized_model(test_input)
    print(f"Prediction for input {test_input.tolist()}: {prediction.argmax(dim=1).item()}")

In [ ]:
# Compare the quantized model with the float one: the size of the saved weights, and
# whether the two agree on the next-token prediction for the same inputs
print(quantized_model)  # Embedding stays float; LSTM and Linear became DynamicQuantized*


def state_dict_mb(m):
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6


eval_input = torch.randint(0, vocab_size, (256, seq_length))
with torch.no_grad():
    fp32_probs = model(eval_input)
    int8_probs = quantized_model(eval_input)

agreement = (fp32_probs.argmax(dim=1) == int8_probs.argmax(dim=1)).float().mean().item()
fp32_mb, int8_mb = state_dict_mb(model), state_dict_mb(quantized_model)
print(f"Weights: {fp32_mb:.2f} MB (fp32) -> {int8_mb:.2f} MB (int8)")
print(f"Top-1 agreement with the float model: {agreement:.1%}")
print(f"Max probability difference: {(fp32_probs - int8_probs).abs().max().item():.2e}")
assert int8_mb < fp32_mb / 2, "the LSTM and Linear weights should shrink about 4x"

In [ ]:
# Look inside: reproduce the quantized fc weight from the float one
w = model.fc.weight.detach()
w_q = quantized_model.fc.weight()  # a quantized tensor: int8 values + scale + zero point
print(w_q.dtype, w_q.qscheme(), f"scale={w_q.q_scale():.3e}", f"zero_point={w_q.q_zero_point()}")

# TODO: symmetric per-tensor qint8 (zero point 0). Compute the scale from w, then the
# int8 values q (as a torch.int8 tensor). PyTorch computes round(w * (1 / scale)),
# half-to-even; round(w / scale) can differ by 1 when w / scale lies within float
# error of a .5 tie.
scale = ...
q = ...

assert abs(scale.item() - w_q.q_scale()) <= 1e-6 * scale.item(), "scale should be max|w| / 127.5"
assert torch.equal(q, w_q.int_repr()), "your int8 values should match the stored ones"
err = (w_q.dequantize() - w).abs().max().item()
assert err <= scale.item() / 2 * (1 + 1e-4), "rounding error is at most scale / 2"
print(f"max |w - s*q| = {err:.2e} <= s/2 = {scale.item() / 2:.2e}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("quantize-lm", LanguageModel=LanguageModel)